In [ ]:
import os
import requests
import pandas as pd
from entsoe import EntsoePandasClient
import zipfile
import io
import trafilatura
from tqdm import tqdm
from pathlib import Path


# Ключи API загружаются из .env, если он есть.
from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env")



# Временные данные по США

* facets/respondent - единица территорияи
* facets/type -  тут уже что связано с элекричество(например происзводство)
* start= end= дата_начала, дата_конца
* sort column = period - очевидно 
* sort direction = asc-очевидно 
* offset = 0 - начало индекса от которого скачиывваем данные
* length = 5000 -  сколько наюблюдений скачивать( число <=5000)

In [ ]:
API_KEY = os.environ["USA_API"]
BASE_URL = "https://api.eia.gov/v2"

url = f"{BASE_URL}/electricity/rto/region-data/facet/type"

response = requests.get(url,params={"api_key": API_KEY})

response.raise_for_status()
data = response.json()

data

Тут смогли получить названия областей по эоектроэнергии 

In [ ]:
# Исходная выгрузка сохраняется отдельно и не перезаписывает готовый датасет.
import time

# Показатель EIA: D — нагрузка; DF — прогноз; NG — генерация; TI — обмен.
SERIES_TYPE = "DF"
SERIES_FILES = {"D": "demand", "DF": "dayahead_forecast", "NG": "net_generation", "TI": "Total_interchange"}
if SERIES_TYPE not in SERIES_FILES:
    raise ValueError(f"Неизвестный тип EIA: {SERIES_TYPE}")
from concurrent.futures import ThreadPoolExecutor

API_KEY = os.environ["USA_API"]
BASE_URL = "https://api.eia.gov/v2"

url = f"{BASE_URL}/electricity/rto/region-data/facet/respondent/"
df_init = pd.DataFrame()

response = requests.get(url,timeout = 60,
    params={"api_key": API_KEY})
response.raise_for_status()
all_respondent = response.json()["response"]["facets"]


for name in all_respondent:

    name_id = name["id"]
    url = f"{BASE_URL}/electricity/rto/region-data/data/"
    
    for tries in range(5):
        try:
            response = requests.get(url,timeout=20,
                params={
                    "api_key": API_KEY,
                    "frequency": "hourly",
                    "data[0]": "value",
                    "facets[respondent][]": name_id,"facets[type][]": SERIES_TYPE,
                    "start": "2019-01-01T00","end": "2026-10-02T14",
                    "sort[0][column]": "period","sort[0][direction]": "asc",
                    "offset": 0,"length": 10})

            response.raise_for_status()
            total = int(response.json()["response"]["total"])
            break

        except Exception:
            if tries == 4:
                raise
            time.sleep(2 * tries)


    def download(offset):
        for tries in range(5):
            try:
                response = requests.get(url,timeout = 20,                                                                                  
                    params={"api_key": API_KEY,
                        "frequency": "hourly",
                        "data[0]": "value",
                        "facets[respondent][]": name_id,"facets[type][]": SERIES_TYPE,
                        "start": "2019-01-01T00","end": "2026-10-02T14",
                        "sort[0][column]": "period","sort[0][direction]": "asc",
                        "offset": offset,"length": 5000})
                response.raise_for_status()
                return pd.DataFrame(response.json()["response"]["data"])
            except Exception:
                if tries == 4:
                    raise
                time.sleep(2*tries)


    with ThreadPoolExecutor(max_workers=5) as executor:

        dfs = executor.map(download,range(0, total, 5000))

        df_init = pd.concat([df_init, *dfs],ignore_index=True)
        print(len(df_init))
output = Path(r"D:\research\datasets_USA") / f"{SERIES_FILES[SERIES_TYPE]}_raw.csv"
if output.exists():
    raise FileExistsError(f"Файл уже существует: {output}")
df_init.to_csv(output)

# Временные данные по Европе 

In [3]:
# Первичная выгрузка ENTSO-E отдельно от готовых CSV.
# Выбери показатель; export/import — физические трансграничные потоки.
DATASET = "generation"  # generation, load, load_forecast, import, export
client = EntsoePandasClient(api_key=os.environ["EU_API"])
start = pd.Timestamp("2019-01-01", tz="Europe/Brussels")
end = pd.Timestamp("2019-01-10", tz="Europe/Brussels")
frames = []
for country in ["FR", "DE_LU", "NL", "BE", "PL", "IT"]:
    if DATASET == "generation":
        df = client.query_generation(country, start=start, end=end)
    elif DATASET == "load":
        df = client.query_load(country, start=start, end=end)
    elif DATASET == "load_forecast":
        df = client.query_load_forecast(country, start=start, end=end)
    elif DATASET in ("import", "export"):
        df = client.query_physical_crossborder_allborders(country, start=start, end=end, export=(DATASET == "export"))
    else:
        raise ValueError(f"Неизвестный показатель: {DATASET}")
    df["country"] = country
    frames.append(df)
    print(country, len(df))



FR 216
DE_LU 864
NL 864
BE 216
PL 216
IT 216


# Восстановление временных рядов BE/NL и UK


In [ ]:
# Восстановление рядов по официальным источникам.
# По умолчанию выполняется только проверка/план, существующие CSV не меняются.
# Чтобы действительно записать данные, добавьте '--apply' к нужной команде.
from pathlib import Path
import subprocess
import sys

project_root = Path(r'D:\research')
script_dir = project_root / 'text_collection_all_needed'

# BE/NL: повторное скачивание физических потоков Elia.
# Без --apply скачивает источники и показывает число изменений.
subprocess.run([sys.executable, str(script_dir / 'restore_elia_flows.py')], cwd=project_root, check=True)

# UK: по умолчанию только показывает план. Для восстановления добавьте --apply.
subprocess.run([sys.executable, str(script_dir / 'rebuild_uk_series.py')], cwd=project_root, check=True)


# Текстовые данные релевантные 

In [ ]:
from pathlib import Path
import subprocess
import sys

# Укажи замкнутый период сбора. Не включай текущий незавершённый месяц.
START_MONTH = '2019-01'
END_MONTH = '2026-10'
WORKERS = 4
RUN_VERSION = 'v4'  # меняй только при намеренном новом корпусе

def month_range(start_month, end_month):
    start_year, start_number = map(int, start_month.split('-'))
    end_year, end_number = map(int, end_month.split('-'))
    months = []
    year, number = start_year, start_number
    while (year, number) <= (end_year, end_number):
        months.append(f'{year:04d}-{number:02d}')
        year, number = (year + 1, 1) if number == 12 else (year, number + 1)
    return months

project_root = Path.cwd()
while not (project_root / 'text_collection_all_needed' / 'collect_all_countries_electricity.py').is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError('Не найден text_collection_all_needed/collect_all_countries_electricity.py')
    project_root = project_root.parent

months = month_range(START_MONTH, END_MONTH)
run_name = f'electricity-{START_MONTH}-to-{END_MONTH}-{RUN_VERSION}'
script = project_root / 'text_collection_all_needed' / 'collect_all_countries_electricity.py'
texts_dir = project_root / 'all_countries_electricity_texts'
technical_dir = project_root / 'all_countries_electricity_technical'

command = [
    sys.executable, str(script),
    '--months', *months,
    '--texts-dir', str(texts_dir),
    '--technical-dir', str(technical_dir),
    '--run-name', run_name,
    '--workers', str(WORKERS),
]
print(f'Запускаю полный сбор: {START_MONTH} — {END_MONTH}.')
print(f'Основной корпус: {texts_dir}')
print(f'Технический журнал: {technical_dir / run_name}')
subprocess.run(command, cwd=project_root, check=True)
print('Сбор завершён.')


In [ ]:
from pathlib import Path
import subprocess
import sys

# Сбор государственного и политического контекста. Берём только завершённые месяцы.
START_MONTH = '2019-01'
END_MONTH = '2026-09'
WORKERS = 4
RUN_VERSION = 'v2'  # меняй только для намеренно нового запуска
SOURCE_CANDIDATE_CEILING = 1000  # технический предел одного архива в месяц; это не выборка текстов

def month_range(start_month, end_month):
    start_year, start_number = map(int, start_month.split('-'))
    end_year, end_number = map(int, end_month.split('-'))
    months = []
    year, number = start_year, start_number
    while (year, number) <= (end_year, end_number):
        months.append(f'{year:04d}-{number:02d}')
        year, number = (year + 1, 1) if number == 12 else (year, number + 1)
    return months

project_root = Path.cwd()
while not (project_root / 'text_collection_all_needed' / 'collect_all_countries_government_politics.py').is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError('Не найден text_collection_all_needed/collect_all_countries_government_politics.py')
    project_root = project_root.parent

months = month_range(START_MONTH, END_MONTH)
run_name = f'government-politics-{START_MONTH}-to-{END_MONTH}-{RUN_VERSION}'
script = project_root / 'text_collection_all_needed' / 'collect_all_countries_government_politics.py'
texts_dir = project_root / 'all_countries_government_politics_texts'
technical_dir = project_root / 'all_countries_government_politics_technical'

command = [
    sys.executable, str(script),
    '--months', *months,
    '--texts-dir', str(texts_dir),
    '--technical-dir', str(technical_dir),
    '--run-name', run_name,
    '--workers', str(WORKERS),
    '--max-documents-per-territory-month', '0',  # 0 = все найденные тексты, без семплирования
    '--source-candidate-ceiling', str(SOURCE_CANDIDATE_CEILING),
]
print(f'Запускаю государственно-политический корпус: {START_MONTH} — {END_MONTH}.')
print(f'Основной корпус: {texts_dir}')
print(f'Технический журнал: {technical_dir / run_name}')
subprocess.run(command, cwd=project_root, check=True)
print('Сбор завершён.')


In [ ]:
from pathlib import Path
import subprocess
import sys

# Общий новостной фон: отдельный корпус, не смешанный с энергетикой и политикой.
# От 100 до 150 текстов на страну-месяц: при недоборе месяц не попадёт в корпус.
START_MONTH = '2023-09'
END_MONTH = '2026-10'  # только завершённые месяцы
WORKERS = 4
MAX_TEXTS_PER_TERRITORY_MONTH = 150
MIN_TEXTS_PER_TERRITORY_MONTH = 100
RUN_VERSION = 'v1'

def month_range(start_month, end_month):
    start_year, start_number = map(int, start_month.split('-'))
    end_year, end_number = map(int, end_month.split('-'))
    months = []
    year, number = start_year, start_number
    while (year, number) <= (end_year, end_number):
        months.append(f'{year:04d}-{number:02d}')
        year, number = (year + 1, 1) if number == 12 else (year, number + 1)
    return months

project_root = Path.cwd()
while not (project_root / 'text_collection_all_needed' / 'collect_all_countries_general_news.py').is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError('Не найден text_collection_all_needed/collect_all_countries_general_news.py')
    project_root = project_root.parent

months = month_range(START_MONTH, END_MONTH)
run_name = f'general-news-{START_MONTH}-to-{END_MONTH}-{RUN_VERSION}'
script = project_root / 'text_collection_all_needed' / 'collect_all_countries_general_news.py'
texts_dir = project_root / 'all_countries_general_news_texts'
technical_dir = project_root / 'all_countries_general_news_technical'
government_metadata = project_root / 'all_countries_government_politics_texts' / 'metadata.jsonl'

command = [
    sys.executable, str(script),
    '--months', *months,
    '--texts-dir', str(texts_dir),
    '--technical-dir', str(technical_dir),
    '--government-metadata', str(government_metadata),
    '--run-name', run_name,
    '--workers', str(WORKERS),
    '--max-documents-per-territory-month', str(MAX_TEXTS_PER_TERRITORY_MONTH),
    '--minimum-per-territory-month', str(MIN_TEXTS_PER_TERRITORY_MONTH),
]

print(f'Запускаю общий новостной корпус: {START_MONTH} — {END_MONTH}.')
print(f'Основной корпус: {texts_dir}')
print(f'Технический журнал: {technical_dir / run_name}')
subprocess.run(command, cwd=project_root, check=True)
print('Сбор завершён.')

In [ ]:
from pathlib import Path
import subprocess
import sys

project_root = Path(r'D:\research')
script = project_root / 'text_collection_all_needed' / 'run_weather_full.py'

command = [
    sys.executable, str(script),
    '--start-month', '2019-01',
    '--end-month', '2026-09',
    '--run-version', 'v1',
    '--workers', '4',
]

result = subprocess.run(command, cwd=project_root)

if result.returncode == 0:
    print('Сбор завершён. Проверьте месячные отчёты покрытия.')
else:
    print(
        'Сбор завершился с незавершёнными источниками. '
        'Уже сохранённые данные остаются на месте; '
        'для повтора запустите эту же ячейку с тем же run-version.'
    )